In [1]:
import pandas as pd
import numpy as np

# Load CSV
df = pd.read_csv("ais_dataset.csv")

# Remove invalid speeds
df = df[df["sog"] >= 0]

print(df.head())

   id   ship_name       mmsi  sog    cog  heading  nav_status       lon  \
0   1     WIM KAN  244750263  0.0  360.0      511           0   4.89655   
1   2       PIPKA  244024973  0.0    0.0      511           0   4.94021   
2   3  MSC STELLA  352983000  0.0  135.0      135           5  18.45262   
3   4     KRVE 59  244650597  0.0  360.0      511           0   4.14850   
4   5       WILLI  211741170  0.0  360.0      511           5  10.05072   

        lat  width  length  ship_type  
0  52.37911      5      23         69  
1  52.37191      4      10         69  
2 -33.91383     40     304         71  
3  51.94916      4      10         40  
4  53.52865      6      25         37  


In [2]:
# Create group key
group_cols = ["width", "length", "ship_type"]

# Calculate 99th percentile speed
percentiles = (
    df.groupby(group_cols)["sog"]
    .quantile(0.99)
    .reset_index()
    .rename(columns={"sog": "p99_speed"})
)

# Merge back
df = df.merge(percentiles, on=group_cols, how="left")

In [3]:
from sklearn.ensemble import IsolationForest

# Features
features = ["sog", "width", "length", "ship_type"]

X = df[features]

# Train model
iso = IsolationForest(
    n_estimators=100,
    contamination=0.01,  # ~1% anomalies
    random_state=42
)

df["iso_flag"] = iso.fit_predict(X)

# Convert to readable
df["iso_anomaly"] = df["iso_flag"] == -1

In [4]:
df["p99_anomaly"] = df["sog"] > df["p99_speed"]

In [5]:
df["speed_anomaly"] = (
    df["iso_anomaly"] |
    df["p99_anomaly"]
)

In [6]:
df.to_csv("ship_speed_anomalies.csv", index=False)

print("Anomaly detection completed.")

Anomaly detection completed.


In [10]:
df['speed_anomaly'].value_counts()

speed_anomaly
False    7397
True      603
Name: count, dtype: int64